# Normalization Layers & Dropout from Scratch — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Normalization layers are in nearly every modern network: **BatchNorm** in ResNets and most CNNs, **LayerNorm** in every Transformer (ViT, GPT, the adaLN blocks of DiT), **GroupNorm** in diffusion U-Nets (Stable Diffusion's ResBlocks use `GroupNorm(32, C)`) and in detection/segmentation heads where per-GPU batches are tiny. All three compute the same thing,

$$\hat x = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \qquad y = \gamma \odot \hat x + \beta,$$

and differ only in **which axes** $\mu$ and $\sigma^2$ are taken over. The details — biased vs unbiased variance, running statistics and their momentum, train vs eval behaviour, where $\epsilon$ goes — are exactly what interviewers probe and what silently breaks a model ported between frameworks. **Dropout** is the other layer whose behaviour flips between `model.train()` and `model.eval()`.

Implement all four as `nn.Module`s that match PyTorch exactly: outputs **and** gradients.

| layer | input | statistics over | affine $\gamma, \beta$ shape |
|---|---|---|---|
| `BatchNorm2d(C)` | `(N, C, H, W)` | $(N, H, W)$, one per channel | `(C,)` |
| `LayerNorm(shape)` | `(..., *shape)` | the trailing `shape` dims, one per sample/token | `shape` |
| `GroupNorm(G, C)` | `(N, C, *spatial)` | $(C/G, *\text{spatial})$, one per sample and group | `(C,)` |

---

### Requirements

1. **`BatchNorm2d(num_features, eps=1e-5, momentum=0.1)`** for inputs `(N, C, H, W)`.
   - Learnable `nn.Parameter`s `weight` ($\gamma$, init ones) and `bias` ($\beta$, init zeros), shape `(C,)`. Buffers `running_mean` (zeros) and `running_var` (ones), registered with `self.register_buffer`.
   - **Training** (`self.training == True`): per channel, $\mu_c$ = mean over $(N, H, W)$ and $\sigma_c^2$ = the **biased** variance (divide by $n = N \cdot H \cdot W$); normalize with them. Gradients must flow through $\mu$ and $\sigma^2$.
   - Then, without gradient, update the running stats with PyTorch's convention (momentum weights the **new** batch — the opposite of Keras):
     `running_mean = (1 - momentum) * running_mean + momentum * mean` and `running_var = (1 - momentum) * running_var + momentum * var_unbiased`, where $\sigma^2_{\text{unbiased}} = \sigma^2 \cdot \frac{n}{n-1}$.
   - **Eval**: normalize with `running_mean` / `running_var`, and don't update them.
2. **`LayerNorm(normalized_shape, eps=1e-5)`** — `normalized_shape` is an int or a tuple. Normalize each sample over its trailing `len(normalized_shape)` dims (biased variance). `weight` (ones) / `bias` (zeros) have shape `normalized_shape` (elementwise affine). Same behaviour in train and eval.
3. **`GroupNorm(num_groups, num_channels, eps=1e-5)`** — input `(N, C, *spatial)` with any number of spatial dims. Split the channels into `num_groups` groups of $C/G$ **consecutive** channels; normalize each (sample, group) over its channels and all spatial positions (biased variance). `weight` / `bias` are **per channel**, shape `(C,)`. Same in train and eval. ($G = 1$: LayerNorm over $(C, H, W)$ with per-channel affine; $G = C$: InstanceNorm.)
4. **`Dropout(p=0.5)`** — *inverted* dropout. Training: zero each element independently with probability `p` and scale the survivors by $1/(1-p)$, so $\mathbb{E}[\text{out}] = x$; draw a fresh mask on every call; don't modify the input in place. Eval: return the input unchanged. Edge cases: `p = 0` is the identity; `p = 1` returns all zeros (no NaN).
5. All four use `self.training` (set by `.train()` / `.eval()`), preserve the input's shape and dtype, keep $\epsilon$ **inside** the square root, use `self.eps`, and work after `.double()` (the grader compares in float64).

---

### Constraints

- ✅ Tensor ops: `mean`, `var(..., unbiased=False)` (or `correction=0`), `sqrt` / `rsqrt`, `reshape`, broadcasting, `torch.rand_like` / `torch.bernoulli`.
- ❌ No `F.batch_norm`, `F.layer_norm`, `F.group_norm`, `F.instance_norm`, `F.dropout` or their `nn.` module versions (the grader detects calls to them).
- ❌ Don't `.detach()` the batch statistics used for normalization — only the running-stat update is gradient-free.

---

<details>
  <summary>💡 Hint</summary>

  - For each layer, write down the reduction axes for an `(N, C, H, W)` input. Use `keepdim=True`, or view a `(C,)` vector as `(1, C, 1, 1)`, so broadcasting lines up.
  - `torch.var` is *unbiased* by default. Which variance makes $\hat x$ have exactly unit variance over the reduced axes?
  - The running-stat update is bookkeeping, not part of the model's differentiable function: what context manager makes that explicit?
  - GroupNorm: a single `reshape` of the input turns it into "normalize over the last dimension".
  - Dropout: which comparison between `torch.rand_like(x)` and `p` is true with probability $1 - p$? What happens to $x/(1-p)$ when $p = 1$?

</details>

---

In [ ]:
import torch
import torch.nn as nn

In [ ]:
torch.manual_seed(0)
feature_map = torch.randn(8, 6, 5, 5) * 2 + 1     # (N, C, H, W): a CNN activation
tokens = torch.randn(4, 10, 16) * 3 - 2           # (B, T, D): a transformer activation


def compare(mine, ref, x, name, seed=0):
    """Give both modules the same random affine params, then compare the output and the
    gradients w.r.t. input / weight / bias for a random upstream gradient, in float64."""
    mine, ref = mine.double(), ref.double()
    g = torch.Generator().manual_seed(seed)
    with torch.no_grad():
        for pname in ("weight", "bias"):
            q = getattr(ref, pname)
            q.copy_(torch.randn(q.shape, generator=g, dtype=torch.float64) * 0.5
                    + (1.0 if pname == "weight" else 0.0))
            getattr(mine, pname).copy_(q)
    x = x.double()
    gy = torch.randn(x.shape, generator=g, dtype=torch.float64)   # NOT ones: d sum(BN(x))/dx ~ 0
    results = []
    for m in (mine, ref):
        m.weight.grad = m.bias.grad = None
        xx = x.clone().requires_grad_()
        y = m(xx)
        y.backward(gy)
        results.append((y.detach(), xx.grad, m.weight.grad, m.bias.grad))
    for what, a, b in zip(("output", "d/dx", "d/dweight", "d/dbias"), *results):
        assert a is not None, f"{name}: {what} is None"
        assert torch.allclose(a, b, rtol=1e-6, atol=1e-8), \
            f"{name}: {what} mismatch (max diff {(a - b).abs().max().item():.2e})"
    print(f"{name}: output and gradients match PyTorch")

In [ ]:
class BatchNorm2d(nn.Module):
    """Batch normalization for (N, C, H, W) inputs, matching nn.BatchNorm2d."""

    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1):
        super().__init__()
        self.eps, self.momentum = eps, momentum
        self.weight = nn.Parameter(torch.ones(num_features))     # gamma
        self.bias = nn.Parameter(torch.zeros(num_features))      # beta
        # Buffers: part of state_dict() and moved by .to(device)/.double(), but not trained.
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            mean = x.mean(dim=(0, 2, 3))                          # (C,)
            var = x.var(dim=(0, 2, 3), unbiased=False)            # biased: what we normalize with
            n = x.numel() // x.shape[1]                           # N * H * W values per channel
            with torch.no_grad():                                 # bookkeeping, not part of the graph
                unbiased = var * n / max(n - 1, 1)                # Bessel's correction
                self.running_mean.mul_(1 - self.momentum).add_(self.momentum * mean)
                self.running_var.mul_(1 - self.momentum).add_(self.momentum * unbiased)
        else:
            mean, var = self.running_mean, self.running_var
        shape = (1, -1, 1, 1)                                     # broadcast (C,) over N, H, W
        x_hat = (x - mean.view(shape)) / torch.sqrt(var.view(shape) + self.eps)
        return x_hat * self.weight.view(shape) + self.bias.view(shape)

In [ ]:
class LayerNorm(nn.Module):
    """Layer normalization over the trailing `normalized_shape` dims, matching nn.LayerNorm."""

    def __init__(self, normalized_shape, eps: float = 1e-5):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(self.normalized_shape))
        self.bias = nn.Parameter(torch.zeros(self.normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dims = tuple(range(-len(self.normalized_shape), 0))       # e.g. (-1,) or (-3, -2, -1)
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, unbiased=False, keepdim=True)
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias

In [ ]:
class GroupNorm(nn.Module):
    """Group normalization for (N, C, *spatial) inputs, matching nn.GroupNorm."""

    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5):
        super().__init__()
        if num_channels % num_groups != 0:
            raise ValueError("num_channels must be divisible by num_groups")
        self.num_groups, self.num_channels, self.eps = num_groups, num_channels, eps
        self.weight = nn.Parameter(torch.ones(num_channels))      # per CHANNEL, not per group
        self.bias = nn.Parameter(torch.zeros(num_channels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        N, C = x.shape[:2]
        g = x.reshape(N, self.num_groups, -1)       # group k = channels [k*C/G, (k+1)*C/G) + all positions
        mean = g.mean(dim=-1, keepdim=True)
        var = g.var(dim=-1, unbiased=False, keepdim=True)
        x_hat = ((g - mean) / torch.sqrt(var + self.eps)).reshape(x.shape)
        shape = (1, C) + (1,) * (x.dim() - 2)
        return x_hat * self.weight.view(shape) + self.bias.view(shape)

In [ ]:
class Dropout(nn.Module):
    """Inverted dropout, matching nn.Dropout."""

    def __init__(self, p: float = 0.5):
        super().__init__()
        if not 0.0 <= p <= 1.0:
            raise ValueError("p must be in [0, 1]")
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0.0:
            return x
        if self.p == 1.0:
            return torch.zeros_like(x)                  # x * 0 / 0 would be NaN
        keep = torch.rand_like(x) >= self.p             # P(keep) = 1 - p
        return x * keep / (1.0 - self.p)                # E[out] = x

In [ ]:
# Test 1 — BatchNorm2d in training mode: output and all gradients
for shape in [(8, 6, 5, 5), (2, 3, 1, 1)]:
    compare(BatchNorm2d(shape[1]), nn.BatchNorm2d(shape[1]), torch.randn(shape) * 2 + 1,
            f"BatchNorm2d train {shape}")

# running stats after several batches of different shapes, then eval mode
mine, ref = BatchNorm2d(6, momentum=0.3).double(), nn.BatchNorm2d(6, momentum=0.3).double()
for step in range(5):
    xb = torch.randn(2 + step, 6, 4, 3 + step, dtype=torch.float64) * (1 + step) + step
    mine(xb), ref(xb)
assert torch.allclose(mine.running_mean, ref.running_mean), "running_mean differs from PyTorch"
assert torch.allclose(mine.running_var, ref.running_var), "running_var differs (unbiased variance? momentum convention?)"
print("running stats match after 5 steps")
mine.eval(), ref.eval()
compare(mine, ref, feature_map, "BatchNorm2d eval (running stats)")
assert "running_mean" in dict(mine.named_buffers()), "use register_buffer for running stats"

In [ ]:
# Test 2 — LayerNorm and GroupNorm
compare(LayerNorm(16), nn.LayerNorm(16), tokens, "LayerNorm(16) on (B, T, D)")
compare(LayerNorm((10, 16)), nn.LayerNorm((10, 16)), tokens, "LayerNorm((10, 16))")
compare(LayerNorm((6, 5, 5)), nn.LayerNorm((6, 5, 5)), feature_map, "LayerNorm((C, H, W))")
for G in (1, 2, 3, 6):
    compare(GroupNorm(G, 6), nn.GroupNorm(G, 6), feature_map, f"GroupNorm({G}, 6)")
compare(GroupNorm(2, 4), nn.GroupNorm(2, 4), torch.randn(3, 4, 7), "GroupNorm(2, 4) on (N, C, L)")
# tiny variance: eps placement matters
compare(LayerNorm(16, eps=1e-3), nn.LayerNorm(16, eps=1e-3), tokens * 1e-3, "LayerNorm, tiny-variance input")

In [ ]:
# Test 3 — Dropout
torch.manual_seed(0)
p = 0.3
drop = Dropout(p)
x = torch.randn(400, 500)
out = drop(x)                                       # modules start in training mode
kept = out != 0
print(f"keep rate {kept.float().mean().item():.4f} (expected {1 - p})")
assert abs(kept.float().mean().item() - (1 - p)) < 0.01
assert torch.allclose(out[kept], x[kept] / (1 - p)), "survivors must be scaled by 1 / (1 - p)"
assert abs(drop(torch.ones(400, 500)).mean().item() - 1.0) < 0.02, "E[dropout(x)] must equal x"
assert not torch.equal(drop(x) != 0, kept), "each call must draw a fresh mask"
drop.eval()
assert torch.equal(drop(x), x), "eval mode must be the identity"
assert torch.equal(Dropout(0.0)(x), x), "p = 0 must be the identity"
z = Dropout(1.0)(x)
assert torch.equal(z, torch.zeros_like(x)) and not z.isnan().any(), "p = 1 must give zeros, not NaN"
print("Dropout OK")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("normalization-layers", BatchNorm2d=BatchNorm2d, LayerNorm=LayerNorm, GroupNorm=GroupNorm, Dropout=Dropout)

### Demo: folding BatchNorm into the preceding convolution

At inference BN is a fixed per-channel affine map, so it can be merged into the conv before it. With
$s_c = \gamma_c / \sqrt{\sigma^2_c + \epsilon}$ (running variance):

$$\mathrm{BN}(W * x + b) = s \odot (W * x + b - \mu) + \beta \;=\; \underbrace{(s \odot W)}_{W'} * x + \underbrace{s \odot (b - \mu) + \beta}_{b'},$$

i.e. scale output-channel $c$'s filter by $s_c$. This is a standard deployment step (TensorRT, ONNX optimizers, quantization pipelines) and removes the BN
layer entirely. It is only possible because BN in eval mode does not depend on the input — LayerNorm/GroupNorm cannot be folded this way.

In [ ]:
torch.manual_seed(0)
conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
bn = BatchNorm2d(8)
with torch.no_grad():
    bn.weight.copy_(torch.rand(8) + 0.5)
    bn.bias.copy_(torch.randn(8))
    for _ in range(20):                                    # accumulate running statistics
        bn(conv(torch.randn(16, 3, 8, 8) * 2 + 1))
bn.eval()

scale = bn.weight / torch.sqrt(bn.running_var + bn.eps)    # (C_out,)
fused = nn.Conv2d(3, 8, kernel_size=3, padding=1)
with torch.no_grad():
    fused.weight.copy_(conv.weight * scale.view(-1, 1, 1, 1))
    fused.bias.copy_((conv.bias - bn.running_mean) * scale + bn.bias)

x = torch.randn(4, 3, 8, 8)
with torch.no_grad():
    diff = (fused(x) - bn(conv(x))).abs().max().item()
print(f"max |conv+BN - folded conv| = {diff:.2e}")
assert diff < 1e-4

## Why it works

**One formula, three choices of axes.** $\hat x = (x - \mu)/\sqrt{\sigma^2 + \epsilon}$ followed by $\gamma \hat x + \beta$. BatchNorm averages over
$(N, H, W)$ — statistics shared across the batch, one per channel; LayerNorm over each sample's trailing feature dims; GroupNorm over each sample's
$C/G$-channel groups. $\gamma, \beta$ let the network undo the normalization if that is optimal, so expressivity is not lost.

**Biased vs unbiased variance.** Normalization uses the biased (maximum-likelihood) variance: that is what makes $\hat x$ have exactly zero mean and unit
variance over the reduced axes. `running_var` is an estimate of the *population* variance used at test time, so PyTorch applies Bessel's correction
$n/(n-1)$ before averaging it in. With $n = N H W$ large the two barely differ, but for small $n$ (BN after a 1×1 spatial map, or tiny batches) it matters —
and getting it wrong breaks parity with pretrained checkpoints.

**Running statistics** are an exponential moving average with weight `momentum` on the newest batch: $r_t = (1-m)\,r_{t-1} + m\,s_t$. They are buffers —
saved in `state_dict`, moved by `.to()`, not touched by the optimizer — and updated under `no_grad` because they are not part of the differentiated function.

**Gradients through the statistics.** Because $\mu$ and $\sigma^2$ depend on every element, BN's input gradient is
$\frac{\partial L}{\partial x} = \frac{\gamma}{\sqrt{\sigma^2+\epsilon}}\big(g - \overline{g} - \hat x\,\overline{g \hat x}\big)$ (bars = means over the reduced
axes, $g = \partial L/\partial y$): the gradient is centred and has its component along $\hat x$ removed. Detaching $\mu, \sigma^2$ drops the last two terms,
which is why the checks use a random upstream gradient — with $g = 1$ the true input gradient is exactly 0 and such bugs would hide. **$\epsilon$ inside the
root** bounds $1/\sqrt{\sigma^2+\epsilon} \le 1/\sqrt\epsilon$ even for constant channels.

**Inverted dropout.** With mask $m \sim \mathrm{Bernoulli}(1-p)$, $\mathbb{E}[m x/(1-p)] = x$, so the expected activation is unchanged and inference needs no
rescaling: eval is the identity. ("Classic" dropout instead multiplies by $1-p$ at test time.)

## Complexity

Every layer is $O(\text{numel})$ time with two reductions (mean, variance) and one elementwise pass — memory-bandwidth bound, which is why fused kernels
(apex/Triton LayerNorm, cuDNN BN) matter in practice. BN keeps $O(C)$ extra state; Dropout's backward needs the mask (often stored as 1 bit/element or
regenerated from the RNG seed).

## Common bugs

- Normalizing with the unbiased variance (`torch.var` default), or updating `running_var` with the biased one.
- Keras-style momentum: `running = momentum * running + (1 - momentum) * batch`.
- Using batch statistics in eval mode (or forgetting `model.eval()`): predictions depend on what else is in the batch.
- Running stats stored as plain tensor attributes instead of `register_buffer` → not in `state_dict`, not moved to GPU.
- Computing the batch statistics under `no_grad` / `.detach()` → wrong input gradient.
- The opposite mistake: updating the running stats *outside* `no_grad` (and without `.detach()`) — the buffers join the autograd
  graph, keep every batch's graph alive, and a later eval-mode backward fails ("Trying to backward through the graph a second time").
- BN reducing over dim 0 only (a per-pixel norm) or LayerNorm over only the last dim when `normalized_shape` has several dims.
- GroupNorm with a strided grouping (`view(N, C // G, G, ...)`) or a per-group instead of per-channel affine.
- `sqrt(var) + eps` instead of `sqrt(var + eps)`, or a hard-coded eps.
- Dropout keeping elements with probability `p`, not scaling by $1/(1-p)$, scaling at eval, or producing NaN at `p = 1`.

## Interview follow-ups

1. **Why does BN behave differently at train and test time?** In training each output depends on the rest of the batch (the statistics), which also injects noise that regularizes; at test time predictions must be deterministic and per-example, so BN uses population estimates (the EMA). Train/test skew appears when the EMA is stale or the data distribution shifts — fix with "precise BN" (re-estimate stats with a pass over data) or by recalibrating on target data.
2. **What goes wrong with small batches, and how does GroupNorm help?** With 1–2 images per GPU (detection, segmentation, high-resolution diffusion) the batch statistics are noisy and differ from the running ones, so accuracy collapses. GroupNorm (Wu & He, 2018) computes statistics per sample, so it is independent of batch size and identical at train and test. Alternatives: SyncBatchNorm across GPUs, frozen BN when fine-tuning, BatchRenorm.
3. **Why BN in CNNs but LayerNorm in Transformers?** Sequences have variable length and padding, and autoregressive decoding processes one token at a time — batch statistics across tokens/sequences are ill-defined and would leak information between examples. LayerNorm normalizes each token over its own features and behaves identically in training and generation. Many LLMs use RMSNorm (no mean subtraction, no bias) for speed; ConvNeXt also moved CNNs to LayerNorm.
4. **How do you fold BN into the preceding conv for inference?** With $s = \gamma/\sqrt{\sigma^2_{\text{run}}+\epsilon}$: $W' = s_c \cdot W_c$ per output channel, $b' = s\,(b - \mu_{\text{run}}) + \beta$ (see the demo). Zero cost at inference; impossible for LN/GN, whose statistics depend on the input.
5. **In what sense is dropout an ensemble?** Each mask selects one of $2^n$ weight-sharing "thinned" sub-networks; training optimizes their average. Using the full network with expectation-preserving scaling at test time approximates the (geometric-mean) ensemble prediction — exactly for a single linear layer. MC-dropout keeps it on at test time and uses the spread of predictions as an uncertainty estimate.
6. **Where should dropout and BN go relative to each other?** Dropout before BN causes a "variance shift": BN's running variance is estimated with dropout noise but applied without it (Li et al., 2019). Common practice is to put dropout after the last BN (e.g. before the classifier) or to rely on BN/weight decay alone in CNNs.